# Приведение к стационарности

## Подготовка

In [5]:
# Handling data
import pandas as pd
import numpy as np

# Visualizations
import matplotlib.pyplot as plt
import plotly.express as px

# Stationarity tests
from statsmodels.tsa.stattools import adfuller, kpss

# Autocorr test
from statsmodels.stats.diagnostic import acorr_ljungbox

# Autocorrelation analysis
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ARIMA models
from statsmodels.tsa.arima.model import ARIMA

# Decomposition
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL, MSTL

import warnings
warnings.filterwarnings("ignore")

In [6]:
!pip install pmdarima

# auto ARIMA
import pmdarima as pm
from pmdarima import auto_arima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 34.1 MB/s eta 0:00:00


In [8]:
def adf_test(timeseries):
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)

In [9]:
def kpss_test(timeseries):
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto")
    kpss_output = pd.Series(
        kpsstest[0:3], index=["Test Statistic", "p-value", "Lags Used"]
    )
    for key, value in kpsstest[3].items():
        kpss_output["Critical Value (%s)" % key] = value
    print(kpss_output)

## Данные

Мы будем анализировать цены закрытия акций. Данные можно скачать [здесь](https://www.nasdaq.com/market-activity/stocks/msft/historical?page=1&rows_per_page=10&timeline=y5). Не забудьте переименовать файл в *'microsoft_new.csv'*.

Рассмотрим акции Microsoft за 5 лет: у нас 1255 наблюдений с 18.11.2020 по 17.11.2025.

In [10]:
stocks = pd.read_csv('microsoft_new.csv', parse_dates=['Date'])

In [11]:
stocks.head()

,Date,Close/Last,Volume,Open,High,Low
0,2025-11-17,$507.49,19092750,$508.45,$512.12,$504.91
1,2025-11-14,$510.18,28505750,$498.23,$511.60,$497.44
2,2025-11-13,$503.29,25273110,$510.31,$513.50,$501.29
3,2025-11-12,$511.14,26574850,$509.355,$511.67,$499.1201
4,2025-11-11,$508.68,17980020,$504.80,$509.60,$502.3488


Немного предобработки.

In [12]:
stocks.rename(columns={'Date':'ds', 'Close/Last':'y'}, inplace=True)
stocks['y'] = stocks['y'].apply(lambda x: x[1:]).astype(float)

stocks = stocks[['ds','y']].sort_values(by='ds').set_index('ds')

In [13]:
stocks.head()

,y
ds,
2020-11-18,211.08
2020-11-19,212.42
2020-11-20,210.39
2020-11-23,210.11
2020-11-24,213.86


Построим график данных.

In [14]:
fig = px.line(stocks, x=stocks.index, y=stocks['y'],
              title="Microsoft closing prices (daily)")

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

На занятиях 3–4 мы обсуждали модели **ARMA**, которые работают только со **стационарными** временными рядами.

Понимание и применение методов приведения к стационарности критически важно при анализе временных рядов: они переводят данные в формат, который соответствует предпосылкам многих статистических и ML-моделей, что делает выводы точнее и понятнее.

В уроках 3–4 мы уже говорили о приведении к стационарности с помощью дифференцирования. Давайте повторим этот подход и перейдём к другим техникам.

## Дифференцирование


Дифференцирование — популярный способ сделать временной ряд стационарным, убрав тренды или сезонность. Особенно полезно, когда данные демонстрируют меняющиеся во времени паттерны, например рост или падение. Операция заключается в вычитании предыдущего значения из текущего (или значения через сезонный лаг), что стабилизирует среднее и удаляет тренд.

Порядок дифференцирования показывает, сколько раз применена эта операция. Обычно достаточно дифференцирования первого порядка.

**Формула** (1-й порядок)

$ΔY_t = Y_t - Y_{t-1}$ или $y'_t = y_t - y_{t-1}$

In [15]:
stocks['diff'] = stocks['y'].diff()
stocks.head()

,y,diff
ds,,
2020-11-18,211.08,NaN
2020-11-19,212.42,1.34
2020-11-20,210.39,-2.03
2020-11-23,210.11,-0.28
2020-11-24,213.86,3.75


Посмотрим, сделало ли дифференцирование первого порядка наш ряд стационарным.

In [16]:
fig = px.line(stocks, x=stocks.index, y=stocks['diff'],
              title="Microsoft closing prices (daily) - diff(1))")

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Дифференцированный ряд выглядит стационарным. Но важно подтвердить это тестом. Здесь достаточно ADF-теста, потому что визуально видно: после дифференцирования ряд должен быть стационарным.

In [17]:
adf_test(stocks['diff'].dropna())

Results of Dickey-Fuller Test:
Test Statistic                  -22.167965
p-value                           0.000000
#Lags Used                        2.000000
Number of Observations Used    1251.000000
Critical Value (1%)              -3.435588
Critical Value (5%)              -2.863853
Critical Value (10%)             -2.568002
dtype: float64


<font color='green'>**p-value <= уровень значимости (по умолчанию 0.05)**</font> ⇒ отвергаем H(0), значит ряд стационарен (ADF).

Но как вернуть исходные значения? Один из способов — использовать `cumsum`.

In [18]:
y_0 = stocks['y'].iloc[0] # берем первое значение как отправную точку
stocks['y_restored_diff'] = np.cumsum(stocks['diff'])+y_0
stocks

,y,diff,y_restored_diff
ds,,,
2020-11-18,211.08,NaN,NaN
2020-11-19,212.42,1.34,212.42
2020-11-20,210.39,-2.03,210.39
2020-11-23,210.11,-0.28,210.11
2020-11-24,213.86,3.75,213.86
...,...,...,...
2025-11-11,508.68,2.68,508.68
2025-11-12,511.14,2.46,511.14
2025-11-13,503.29,-7.85,503.29


Теперь построим восстановленные значения и убедимся, что они совпадают с исходными.

In [19]:
fig = px.line(title="Microsoft closing prices (daily) - diff(1)")
fig.add_scatter(x=stocks.index, y=stocks['y'], mode='lines', name='original y', line=dict(color='silver'))
fig.add_scatter(x=stocks.index, y=stocks['y_restored_diff'], mode='lines', name='restored y', line=dict(color='red'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Иногда дифференцированный ряд всё ещё выглядит нестационарным, и тогда приходится применять дифференцирование второго порядка. На практике почти никогда не нужно идти дальше второго порядка.

В нашем примере второго порядка не требуется, но ниже показано, как его выполнить при необходимости.



In [20]:
stocks['y'].diff().diff()

,y
ds,
2020-11-18,NaN
2020-11-19,NaN
2020-11-20,-3.37
2020-11-23,1.75
2020-11-24,4.03
...,...
2025-11-11,-6.50
2025-11-12,-0.22
2025-11-13,-10.31


## Процентный прирост (percent change)

Процентный прирост очень похож на показатели MoM и YoY, о которых мы говорили на уроках 1–2.

**Формула**

pct\_change$_t = \frac{Y_t - Y_{t-1}}{Y_{t-1}} \times 100$

In [21]:
stocks['pct_change'] = stocks['y'].pct_change()*100
stocks.head()

,y,diff,y_restored_diff,pct_change
ds,,,,
2020-11-18,211.08,NaN,NaN,NaN
2020-11-19,212.42,1.34,212.42,0.634830
2020-11-20,210.39,-2.03,210.39,-0.955654
2020-11-23,210.11,-0.28,210.11,-0.133086
2020-11-24,213.86,3.75,213.86,1.784779


Посмотрим, сделало ли преобразование через процентный прирост наш ряд стационарным.

In [22]:
fig = px.line(stocks, x=stocks.index, y=stocks['pct_change'],
              title="Microsoft closing prices (daily) - pct_change")

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Также проверим это с помощью ADF-теста.

In [23]:
adf_test(stocks['pct_change'].dropna())

Results of Dickey-Fuller Test:
Test Statistic                  -22.667553
p-value                           0.000000
#Lags Used                        2.000000
Number of Observations Used    1251.000000
Critical Value (1%)              -3.435588
Critical Value (5%)              -2.863853
Critical Value (10%)             -2.568002
dtype: float64


<font color='green'>**p-value <= уровень значимости (по умолчанию 0.05)**</font> ⇒ отвергаем H(0), значит ряд стационарен (ADF).

Как восстановить исходный ряд?

In [24]:
y_0 = stocks['y'].iloc[0]
stocks['y_restored_pct_change'] = stocks['y'].shift(1)* (1 + stocks['pct_change'] / 100)
stocks

,y,diff,y_restored_diff,pct_change,y_restored_pct_change
ds,,,,,
2020-11-18,211.08,NaN,NaN,NaN,NaN
2020-11-19,212.42,1.34,212.42,0.634830,212.42
2020-11-20,210.39,-2.03,210.39,-0.955654,210.39
2020-11-23,210.11,-0.28,210.11,-0.133086,210.11
2020-11-24,213.86,3.75,213.86,1.784779,213.86
...,...,...,...,...,...
2025-11-11,508.68,2.68,508.68,0.529644,508.68
2025-11-12,511.14,2.46,511.14,0.483605,511.14
2025-11-13,503.29,-7.85,503.29,-1.535783,503.29


In [25]:
fig = px.line(title="Microsoft closing prices (daily) - pct_change")
fig.add_scatter(x=stocks.index, y=stocks['y'], mode='lines', name='original y', line=dict(color='silver'))
fig.add_scatter(x=stocks.index, y=stocks['y_restored_pct_change'], mode='lines', name='restored y', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

При анализе цен акций абсолютная разница (=дифференцирование первого порядка) между ценами мало о чём говорит и обладает плохими статистическими свойствами. Поэтому эту разницу обычно делят на цену предыдущего периода. Так мы получаем доходности, с которыми удобно работать и с точки зрения теории (почти вся финансовая теория строится на доходностях), и с точки зрения моделей (доходности имеют хорошие статистические свойства). По своей сути доходности - это процентные приросты.

## Декомпозиция

Декомпозиция временного ряда — это разложение ряда на компоненты. Цель — лучше понять структуру данных, которая обычно включает:

* Тренд: долгосрочное направление движения ряда.
* Сезонность: повторяющиеся паттерны или циклы на регулярных интервалах.
* Остатки (шум): случайные колебания, остающиеся после удаления тренда и сезонности.

Такой подход помогает осмысленно анализировать временные ряды и находить проблемы при анализе и прогнозировании. У любого ряда есть уровень и шум; тренд и сезонность могут присутствовать или отсутствовать. Компоненты можно рассматривать как аддитивные или мультипликативные.

* **Аддитивная модель**:

  $y(t) = Trend + Seasonality + Residuals$

* **Мультипликативная модель**:

  $y(t) = Trend * Seasonality * Residuals$

Обратите внимание: остатки могут также называться «remainder».


### Классическая декомпозиция

*Алгоритм*

* Скользящее среднее используется для фильтрации трендово-циклической компоненты исходного ряда.
* Детрендированные значения для каждого сезонного шага усредняются, формируя точки сезонной компоненты.
* Эти точки центрируются вокруг нуля и объединяются в сезонную компоненту.
* Всё, что остаётся после вычитания тренда и сезонности из исходных данных, считается остатком.

*Ограничения*

* Не справляется со сложной сезонностью (несколько сезонностей или нерегулярные сезонные паттерны).
* Чувствителен к выбросам.
* Предполагает фиксированную длину сезонного цикла, что подходит не всем датасетам.
* Плохо работает с нелинейными трендами.
* Считает сезонность постоянной от года к году (или другого периода), что редко верно в реальных задачах.
* Любые вариации, не объяснённые трендом и сезонностью, остаются в остатках и трудно интерпретируются.

*Подходит для*

* Простых сезонных паттернов.
* Около линейных или слегка меняющихся трендов.
* Достаточного объёма данных для оценки сезонных индексов.

Классическая декомпозиция даёт базовое представление о структуре временного ряда — это её плюс и минус одновременно. Исторически этот метод помог создать более сложные техники. Он полезен как стартовый анализ, но для сложных рядов лучше использовать более комплексные методы.

См. также [Hyndman](https://otexts.com/fpp3/classical-decomposition.html).

Попробуем декомпозицию с разной длиной сезона (то есть разными **периодами**).

In [ ]:
res = seasonal_decompose(stocks['y'],  period=7)

fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

res.resid.plot()

In [ ]:
res.seasonal[:28*4].plot()

In [ ]:
stl = seasonal_decompose(stocks['y'],  period=28)

fig = stl.plot()
fig.set_size_inches(10, 6)
plt.show()

stl.resid.plot()

In [ ]:
stl = seasonal_decompose(stocks['y'],  period=365)

fig = stl.plot()
fig.set_size_inches(10, 6)
plt.show()

stl.resid.plot()

#### Остатки после декомпозиции

Каждый раз остатки выводились отдельным графиком. Зачем?

Идеально, если после удаления тренда и сезонности остатки ведут себя как белый шум.

🌵 **Что такое белый шум?**

Белый шум имеет постоянное среднее, постоянную дисперсию и не коррелирует во времени (нет паттернов и зависимостей). Если остатки напоминают белый шум, значит декомпозиция хорошо выделила тренд и сезонность. Если же в остатках остаются паттерны, это сигнал, что мы что-то недоучли — например, тренд или сезонность.

Многие методы прогнозирования предполагают, что остатки — белый шум. Если это не так, модель может работать хуже, поскольку оставшиеся паттерны будут смещать прогнозы.

Итак, **чего мы ждём от остатков?**

- **Отсутствия автокорреляции**
  
  Это значит, что декомпозиция успешно убрала все систематические паттерны (тренд, сезонность и прочие зависимости).

- **Стационарности**

  Формально проверка остатков *(после декомпозиции)* на стационарность не обязательна, чтобы говорить об успехе декомпозиции. Но все-таки важно анализировать стационарность остатков.
  
  Стационарные остатки подсказывают, что тренд и сезонность удалены корректно.
  
  Нестационарность может означать:
  1) тренд не удалён до конца;
  2) указан неверный сезонный период;
  3) в данных остались циклические или долгосрочные компоненты.

  При этом в регрессионных моделях с временными рядами остатки обязательно должны быть стационарными, иначе получим ложные зависимости (об этом поговорим позже).


**ИТОГ:**
- Стационарность подтверждает, что в остатках нет эволюционирующих паттернов (например, тренда).
- Отсутствие автокорреляции гарантирует, что все зависимости действительно удалены.

Давайте проверим остатки после декомпозиции на отсутствие автокорреляции (обязательно!) и на стационарность (важно для следующих шагов).

🌵 **Как проверить отсутствие автокорреляции?**

С помощью ACF и PACF:

In [ ]:
# 2 graphs side by side
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# ACF
plot_acf(stl.resid.dropna(), ax=ax[0], lags=20) # nans are only present in the first 3 and the last 3 observations
ax[0].set_title('ACF')

# PACF
plot_pacf(stl.resid.dropna(), ax=ax[1], lags=20) # nans are only present in the first 3 and the last 3 observations
ax[1].set_title('PACF')


plt.tight_layout()
plt.show()

С помощью критерия Льюнга—Бокса:


In [ ]:
acorr_ljungbox(stl.resid.dropna(), return_df=True) # nans are only present in the first the last observations

🌵 **Что скажете про автокорреляцию?**

И ACF/PACF, и тест Льюнга—Бокса показывают, что остатки автокоррелированы. Это плохой знак: вероятно, при декомпозиции мы не учли какую-то трендовую или сезонную зависимость.

🌵 **Как проверить стационарность?**

С помощью ADF-теста:

In [ ]:
adf_test(res.resid.dropna()) # nans are only present in the first 3 and the last 3 observations

К счастью, остатки стационарны.

Подведём итог по остаткам.

Если остатки после декомпозиции **стационарны, но автокоррелированы**, значит, мы удалили долгосрочные тренды и сезонность (есть стационарность), но оставили краткосрочные зависимости или систематические паттерны.

Тренд и сезонная компонента, возможно, не объяснили все паттерны, поэтому в остатках остаётся корреляция. Некоторые динамики ряда не являются периодическими (сезонными) или монотонными (трендовыми), но носят систематический характер — например, краткосрочные циклы.

### Декомпозиция STL

**STL** (Seasonal and Trend decomposition using LOESS) — более гибкий и современный подход, чем классическая декомпозиция. В отличие от классики, где сезонность фиксирована, STL использует локально взвешенную регрессию (LOESS) для оценки тренда и сезонности, поэтому лучше подходит для данных со сложной сезонностью и трендами.

*STL* — продвинутый и устойчивый метод, часто применяемый в бизнес-практиках. *STL* и его расширение *MSTL* — мощные альтернативы классической декомпозиции для реальных временных рядов.

*Алгоритм*

STL работает в двух циклах.
* *Внешний цикл* назначает веса наблюдениям в зависимости от величины остатка.
* Учитываются соседние точки и большим весом наделяются ближайшие, что помогает ловить нелинейные тренды.
* *Внутренний цикл* делит детрендированный ряд на подсерии и сглаживает их LOESS.
* Затем подсерии пропускаются через фильтр низких частот, после чего обновляются сезонность и тренд.
* Последующие итерации уточняют оценки.
* Ключевые параметры: 1) длина сезонного периода; 2) степень сглаживания сезонных подсерий (seasonal window).

*Ограничения*
* Достаточно устойчив к выбросам, но плохо справляется с праздниками и особыми датами.
* Работает только с аддитивной декомпозицией (проблема решается логарифмированием данных перед применением STL).

*Подходит для*

* Сложных и меняющихся сезонных паттернов.
* Шумных данных или данных с выбросами.
* Ситуаций, когда нужно больше контроля над сглаживанием тренда и сезонности.

См. также [Hyndman](https://otexts.com/fpp3/stl.html) и [Демешева](https://www.youtube.com/watch?v=bTmsWGNCLfI).

In [ ]:
stl = STL(stocks['y'],  period=7)
res = stl.fit()

fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

res.resid.plot()

In [ ]:
stl = STL(stocks['y'],  period=28)
res = stl.fit()

fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

res.resid.plot()

In [ ]:
stl = STL(stocks['y'],  period=365)
res = stl.fit()

fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

res.resid.plot()

🌵 **А как обстоят дела с автокорреляцией и стационарностью?**

In [ ]:
acorr_ljungbox(res.resid, return_df=True)

In [ ]:
adf_test(res.resid)

Ситуация чуть хуже: остатки стационарны (но p-value гораздо больше) и автокоррелированы.

🌵 **В чём может быть проблема?**

### Декомпозиция MSTL

Модификация STL под названием **MSTL** (Multi-Scale Seasonal and Trend Decomposition using LOESS) создана для временных рядов, где часто есть несколько сезонностей. MSTL расширяет технику, чтобы работать с несколькими сезонными паттернами в течение года. Метод подбирает поверхность LOESS, а не одну линию в каждый момент, и одновременно извлекает тренд и все сезонности.

*Подходит для*

* Тех же случаев, что и STL, но особенно полезен при нескольких сезонных паттернах.

🌵 **Какие длины сезонных периодов выбрать?**

Посмотрим на данные ещё раз.

🌵 **Что замечаете?**

In [ ]:
stocks['y'].loc['2024-11-01':]

Обратите внимание, что в нашем ряду нет выходных.

* 5 — недельный период
* 20 — месячный
* 252 — годовой

In [ ]:
mstl = MSTL(stocks['y'],  periods=(5, 20, 252))
res = mstl.fit()

fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

res.resid.plot()

🌵 **Как здесь обстоят дела с автокорреляцией и стационарностью?**

In [ ]:
acorr_ljungbox(res.resid, return_df=True)

In [ ]:
adf_test(res.resid)

Остатки остаются стационарными, но, увы, всё ещё автокоррелированы :(

Что делать, **если остатки автокоррелированы?**

Рекомендации:

* Попробуйте изменить сезонные периоды в MSTL.
* Рассмотрите альтернативные методы декомпозиции (STL, классический), иерархические подходы или другие способы приведения к стационарности.
* Проверьте стационарность. Смоделируйте остатки с помощью ARMA (если они стационарны) или ARIMA (если нет; об этом поговорим далее).
* Изучите влияние внешних факторов на поведение цен акций — это может снизить автокорреляцию в остатках.

## Итоги по приведению к стационарности


**Когда применять каждую технику?**

* **Дифференцирование** эффективно удаляет долгосрочные тренды в несезонных данных.
* **Процентное изменение** хорошо работает с финансовыми рядами высокой волатильности.
* **Классическая декомпозиция** подходит для данных без аномалий, с одной сезонностью, постоянной во времени, и с линейным трендом.
* **STL-декомпозиция** хороша для реальных данных, когда есть аномалии, одна сезонность, которая может изменяться, и линейный или нелинейный тренд.
* **MSTL-декомпозиция** нужна для реальных данных, если у ряда несколько сезонных паттернов (и также во всех случаях, где подходит STL).

# Прогнозирование с ARMA

**Формула ARMA(p,q)**

$y_{t} = \alpha_{0} + \alpha_{1}y_{t-1} + ... + \alpha_{p}y_{t-p} + \beta_{1}\epsilon_{t-1} + ... + \beta_{q}\epsilon_{t-q} + \epsilon_{t}$, где $\epsilon_{t}$ — белый шум.

🌵 **Чего требуют модели ARMA от временных рядов?**

Модели ARMA требуют, чтобы временной ряд был стационарным.

Сделаем ряд акций стационарным с помощью дифференцирования. Помните, мы уже это делали: дифференцирование первого порядка лежит в столбце `diff`.

In [ ]:
stocks.head()

Сохраним дифференцированный ряд и уберём `NaN`.

In [ ]:
stocks_diff = stocks[['diff']].dropna().copy()
stocks_diff

## Прогноз in-sample

На занятиях 3–4 мы строили только in-sample прогнозы. Сегодня разберём детали.

Начнём с выбора порядков для модели ARMA(p,q).

🌵 **Как подобрать порядки (p,q) для модели ARMA(p,q)?**

С помощью ACF и PACF:

In [ ]:
# 2 graphs side by side
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# ACF
plot_acf(stocks_diff, ax=ax[0], lags=20) # nans are only present in the first 3 and the last 3 observations
ax[0].set_title('ACF')

# PACF
plot_pacf(stocks_diff, ax=ax[1], lags=20) # nans are only present in the first 3 and the last 3 observations
ax[1].set_title('PACF')


plt.tight_layout()
plt.show()

С помощью AIC:

In [ ]:
best_params = []
for p in np.arange(0,3): # AR with p up to 3 lag
  for q in np.arange(0,3): # MA with q up to 3 lag
    model = ARIMA(stocks_diff, order=(p,0,q)).fit()
    best_params.append({'p':p, 'q':q, 'AIC':model.aic, 'BIC':model.bic})

pd.DataFrame(best_params).sort_values(by='AIC').head()

🌵 **Какие порядки (p,q) подходят лучше всего?**

Возьмём модель ARMA(2,2).

In [ ]:
# Инициализируем ARMA(2,2)
model = ARIMA(stocks_diff, order=(2,0,2)) # orders are (p,d,q), p - AR, q - MA

# Fit the model
model = model.fit()

print(model.summary())

🌵 **Какие коэффициенты оказались значимыми?**

Теперь спрогнозируем $\hat{y_t}$ (будем называть `y_hat`) для нашего ряда $y_t$.

In [ ]:
forecast = model.predict()

plt.plot(stocks_diff,color='red')
plt.plot(forecast,color='blue')
plt.legend(['y','y_hat'])
plt.show()

Но как модель ARMA на самом деле оценивает коэффициенты $\alpha$ и $\beta$?

### Оценка максимального правдоподобия (MLE)


Простой пример (по [Демешеву](https://www.youtube.com/watch?v=2iRIqkm1mug)):

Представьте, что вы приехали в город, пришли на площадь и увидели работающий фонтан. Возможны две гипотезы:

1. Фонтан работает каждый вечер.
2. Фонтан включают раз в год.

Какова вероятность увидеть работающий фонтан в этих случаях?

1. 1
2. 1/365

**MLE советует выбрать гипотезу с максимальной вероятностью.** ⇒ Значит, считаем, что фонтан работает каждый день.


Метод **максимального правдоподобия (MLE)** часто используется для оценки параметров ARMA, потому что он максимизирует вероятность наблюдаемых данных при заданных предположениях модели. Идея такая:

1. Есть неизвестные $\alpha$ и $\beta$.
2. Хотим получить оценки $\hat{\alpha}$ и $\hat{\beta}$.
3. Выбираем такие $\hat{\alpha}$ и $\hat{\beta}$, которые максимизируют вероятность получить наблюдаемые значения $y_t$.

Метод основан на понятии **функции правдоподобия**.



* Функция правдоподобия $L$ для наблюдений $y_1, y_2, ..., y_n$ и параметров $\alpha_1, ..., \alpha_n$, $\beta_1, ..., \beta_n$ равна произведению плотностей вероятностей каждого наблюдения с учётом прошлых данных: $L(\alpha, \beta) = \prod(y_1, y_2, ..., y_n|\alpha, \beta)$

* На практике максимизируют логарифм функции правдоподобия — так проще считать (произведение превращается в сумму).

* Для максимизации лог-правдоподобия используют численные методы оптимизации (например, Ньютон—Рафсон или BFGS), которые итеративно обновляют параметры.

* В итоге получаем набор параметров $\hat{\alpha}$ и $\hat{\beta}$, которые наилучшим образом объясняют наблюдаемые данные.

Когда вы обучаете модель ARMA в библиотеках вроде statsmodels, внутри используется MLE для оценки параметров. Поэтому процесс сводится к указанию порядков (p, q) и вызову `.fit()`: библиотека считает правдоподобие данных при выбранной модели и подбирает параметры.

### Критерии AIC, AICc, BIC

Помните, мы говорили, что AIC и BIC помогают выбрать модель, которая хорошо описывает данные и не содержит лишней сложности.

Эти критерии основаны на концепции правдоподобия.

BIC (информационный критерий Акаике)

$AIC = −2\ln(L)+2(p+q+k+1)$,

где $L$ — максимизированная функция правдоподобия, $k=1$, если $const≠0$, и $k=0$, если $const=0$. Последний член в скобках — число параметров модели (включая $σ^2$, дисперсию остатков — см. матрицу summary после обучения ARMA).

AICc — поправка к AIC, особенно полезная, когда размер выборки $n$ небольшой относительно числа параметров.

$AICc = AIC + \frac{2(p+q+k+1)(p+q+k+2)}{n-p-q-k-2}$

BIC (байесовский информационный критерий) — ещё один инструмент выбора модели, но с более жёстным штрафом за количество параметров. Штраф растёт с размером выборки, поэтому BIC помогает избегать переобучения на больших данных.

$BIC = AIC + (\ln(n)-2)(p+q+k+1)$

Как посчитать AICc?

In [ ]:
# AIC and number of parameters
aic = model.aic
k = len(model.params)  # number of parameters in the model
n = len(stocks_diff)  # sample size

# Calculate AICc
aicc = aic + (2 * k * (k + 1)) / (n - k - 1)

aic, aicc

Эти информационные критерии обычно хорошо помогают выбрать подходящие p и q.

### Остатки модели

**Формула**

$e_t = y_t - \hat{y_t}$

«Остатки» в модели временного ряда — это то, что остаётся после фита модели. Для многих (но не всех) моделей остатки равны разнице между наблюдениями и соответствующими предсказанными значениями.


Хорошая модель прогнозирования даёт остатки со следующими свойствами:

* **Нет автокорреляции**. Если остатки коррелируют, значит, в них осталась полезная информация, которую стоило учесть в модели.
* **Среднее остатка равно нулю**. Иначе прогнозы смещены.

**Любой метод прогнозирования, не удовлетворяющий этим свойствам, можно улучшить. Но даже если метод удовлетворяет этим свойствам, это не значит, что его нельзя сделать лучше.** Для одного и того же набора данных может существовать несколько моделей, отвечающих этим критериям. Проверка свойств нужна, чтобы убедиться, что метод использует всю доступную информацию, но это не лучший способ выбора прогнозной модели.

Теперь проверим остатки.

In [ ]:
forecast = pd.DataFrame(forecast)
forecast.columns = ['diff']
forecast

In [ ]:
residuals = stocks_diff - pd.DataFrame(forecast)

plt.plot(residuals)

Проверим среднее:

In [ ]:
residuals.mean()

Проверим автокорреляцию:

In [ ]:
acorr_ljungbox(residuals, return_df=True)

🌵 **Остатки кажутся хорошими? Что тогда можно сказать о модели?**

Среднее равно 0, автокорреляции нет.

## Прогноз out-of-sample

Наконец, перейдём к прогнозу out-of-sample.

Out-of-sample прогнозирование в ARMA означает предсказания на части данных, которая не использовалась при обучении. Это ключевой шаг, чтобы оценить, насколько модель обобщается на новые данные: мы смотрим, как хорошо ARMA предсказывает будущее, опираясь на прошлое, и проверяем результат на отложенной части выборки.

In [ ]:
train_size = int(len(stocks_diff) * 0.98)
train, test = stocks_diff[:train_size], stocks_diff[train_size:]

In [ ]:
fig = px.line(title="Microsoft closing prices (daily) - diff(1)")
fig.add_scatter(x=train.index, y=train['diff'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['diff'], mode='lines', name='test', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Теперь обучим новую модель на train и построим прогноз с ARMA(2,2).

In [ ]:
# Instantiate the ARMA(2,2) model
model = ARIMA(train, order=(2,0,2)) # orders are (p,d,q), p - AR, q - MA

# Fit the model
model = model.fit()

In [ ]:
forecast_size = len(stocks_diff) - train_size

forecast_size

In [ ]:
y_hat = model.forecast(steps=forecast_size)
y_hat = pd.DataFrame(y_hat)
y_hat['ds'] = test.index # redefine index for forecasts
y_hat = y_hat.set_index('ds')
y_hat

In [ ]:
fig = px.line(title="Microsoft closing prices (daily) - diff(1)")
fig.add_scatter(x=train.index, y=train['diff'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['diff'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=y_hat.index, y=y_hat['predicted_mean'], mode='lines', name='forecast', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Out-of-sample прогнозы ARMA-моделей опираются на среднее, прошлые наблюдения (в нашем случае их 2) и последние известные ошибки (тоже 2). Cтационарные ARMA модели всегда дают долгосрочный прогноз, стремящийся к безусловному среднему ряда. Это нормальное, корректное поведение модели.

Модель MA зависит от пред. ошибок. Как только мы доходим до точки, где реальных ошибок больше нет (конец ряда), модель использует среднее значение предыдущих ошибок. Аналогичная логика и в AR.

Поэтому при долгосрочных прогнозах, выходящих за пределы имеющихся данных, ARMA будет стремиться к среднему ряда: новых ошибок или шоков нет, чтобы смещать прогноз. Это ограничение ARMA моделей, особенно при прогнозировании на много шагов вперёд.

Наконец, вернём временной ряд и прогноз к исходной шкале.

In [ ]:
y_hat.columns = ['diff']
stocks_forecasted = pd.concat([train, y_hat])
stocks_forecasted.head()

In [ ]:
y_0 = stocks['y'].iloc[0] # take the first value as the start
stocks_forecasted['y_restored'] = np.cumsum(stocks_forecasted['diff'])+y_0
stocks_forecasted.head()

In [ ]:
fig = px.line(title="Microsoft closing prices (daily) - diff(1)")
fig.add_scatter(x=stocks.index, y=stocks['y'], mode='lines', name='y', line=dict(color='blue'))
fig.add_scatter(x=stocks_forecasted.index[train_size:], y=stocks_forecasted['y_restored'][train_size:], mode='lines', name='y_hat', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Задание на 20 минут

🌵

**1. Преобразуйте ряд с помощью процентного изменения.**

**2. Подберите подходящие порядки (p,q) для ARMA.**

**3. Выполните прогноз внутри и вне выборки с ARMA (не забудьте сделать сплит ряда).**

**4. Верните результаты к исходному ряду.**

# ARIMA(p,d,q)

ARMA работают только со стационарными рядами, поэтому нам приходится сначала преобразовывать данные (например, дифференцировать), а уже затем строить прогноз.

Но есть модель ARIMA. **ARIMA (AutoRegressive Integrated Moving Average)** — популярная модель прогнозирования, объединяющая три компонента: авторегрессию (AR), дифференцирование (I — интегрирование) и скользящее среднее (MA). Она специально предназначена для **нестационарных временных рядов** и прогнозирует будущие значения, используя прошлые наблюдения и ошибки прогнозов.

**Формула**

$y_{t} = \alpha_{0} + \alpha_{1}y'_{t-1} + ... + \alpha_{p}y'_{t-p} + \beta_{1}\epsilon_{t-1} + ... + \beta_{q}\epsilon_{t-q} + \epsilon_{t}$, где $\epsilon_{t}$ — белый шум, а $y'_t$ — дифференцированный ряд.


Вспомним исходный ряд. Мы знаем, что дифференцирование первого порядка делает его стационарным. Построим модель ARIMA(0,1,1) для этого ряда.

In [ ]:
stocks = stocks['y']

In [ ]:
# Instantiate the MA(1) model
model = ARIMA(stocks, order=(0,1,1)) # orders are (p,d,q), p - AR, q - MA, d - I

# Fit the model
model = model.fit()

print(model.summary())

## Прогноз in-sample

In [ ]:
forecast = model.predict()

plt.plot(stocks,color='red')
plt.plot(forecast,color='blue')
plt.legend(['y','y_hat'])
plt.show()

Приблизим график:

In [ ]:
plt.plot(stocks[-50:],color='red') # lasr 50 observations
plt.plot(forecast[-50:],color='blue')
plt.legend(['y','y_hat'])
plt.show()

## Прогноз out-of-sample

In [ ]:
train_size = int(len(stocks) * 0.98)
train, test = stocks[:train_size], stocks[train_size:]

In [ ]:
fig = px.line(title="Microsoft closing prices (daily) - diff(1)")
fig.add_scatter(x=train.index, y=train, mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test, mode='lines', name='test', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Обучим модель на train и спрогнозируем с ARIMA(0,1,1).

In [ ]:
# Instantiate the ARIMA(0,1,1) model
model = ARIMA(train, order=(0,1,1)) # orders are (p,d,q), p - AR, q - MA, d - I

# Fit the model
model = model.fit()

In [ ]:
forecast_size = len(stocks_diff) - train_size + 1

forecast_size

In [ ]:
y_hat = model.get_forecast(steps=forecast_size)
y_hat = y_hat.predicted_mean
y_hat = pd.DataFrame(y_hat)
y_hat['ds'] = test.index # redefine index for forecasts
y_hat = y_hat.set_index('ds')
y_hat

In [ ]:
fig = px.line(title="Microsoft closing prices (daily) - diff(1)")
fig.add_scatter(x=train.index, y=train, mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test, mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=y_hat.index, y=y_hat['predicted_mean'], mode='lines', name='forecast', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Обратите внимание: здесь модель вернула **predicted_mean**. Это ожидаемое будущие значение — самый надёжный «обычный» прогноз, который даёт ARIMA. Модель предполагает, что будущие значения случайны и подчиняются распределению, а predicted_mean — это математическое ожидание (наиболее вероятная точка). Чаще всего этого достаточно для бизнес-задач, но можно использовать и доверительные интервалы, которые модель также предоставляет.

Важно: критерии AIC, AICc и BIC плохо помогают выбирать порядок дифференцирования (d). Дифференцирование меняет данные, по которым считается правдоподобие, поэтому значения AIC у моделей с разными d некорректно сравнивать. Значит, d выбираем другими методами, а затем используем AIC или BIC для подбора p и q.

Как выбрать d?

**Итеративное дифференцирование:**

1. Начните с $d = 0$ (без дифференцирования). Если ряд уже стационарен, оставляем $d = 0$ и подбираем AR/MA.
2. Если ряд нестационарен, применяем дифференцирование первого порядка ($d = 1$): вычитаем предыдущее значение. Затем снова запускаем ADF-тест.
3. Если и после этого ряд нестационарен, применяем дифференцирование второго порядка ($d = 2$) и снова тестируем.
4. Повторяем процесс, пока не получим стационарность. Обычно хватает $d = 0$, $d = 1$ или $d = 2$.


## Auto-ARIMA (автоматический ARIMA)

**Auto ARIMA (Auto-Regressive Integrated Moving Average)** — автоматизированный алгоритм выбора наилучшей ARIMA-модели для прогнозирования временных рядов. Он упрощает поиск оптимальных параметров p, d, q.

In [ ]:
# Fit Auto ARIMA model
model = auto_arima(train, stepwise=True, trace=True)

# Print the summary of the model
print(model.summary())

# Make a forecast for the next periods
forecast_auto_arima = model.predict(n_periods=forecast_size)
print(forecast_auto_arima)

**Как работает Auto-ARIMA?**

Функция **auto_arima** в библиотеке pmdarima по умолчанию использует пошаговый алгоритм на основе метода Хайндмана—Кхандакара. Он комбинирует тесты на единичный корень, минимизацию AICc и MLE, чтобы получить подходящую ARIMA-модель.

Подробнее см. [Hyndman](https://otexts.com/fpp3/arima-r.html).

In [ ]:
forecast_auto_arima = pd.DataFrame(forecast_auto_arima)
forecast_auto_arima['ds'] = test.index
forecast_auto_arima.set_index('ds', inplace=True)
forecast_auto_arima.columns = ['y_hat']

forecast_auto_arima

In [ ]:
fig = px.line(title="Microsoft closing prices (daily)")
fig.add_scatter(x=train.index, y=train, mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test, mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_auto_arima.index, y=forecast_auto_arima['y_hat'], mode='lines', name='forecast', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Теоретически ручная ARIMA и Auto-ARIMA должны давать похожие прогнозы, но Auto-ARIMA зачастую лучше подбирает параметры, оптимизирует остатки и улучшает модель. Поэтому даже при тех же параметрах результаты могут быть лучше.

Кроме того, в pmdarima Auto-ARIMA автоматически учитывает дрейф и константы при обучении. Даже для дифференцированных рядов (d > 0) она может добавить лёгкий тренд вверх или вниз, если это улучшает качество. Поэтому прогноз может плавно меняться, а не оставаться постоянным.



Задание на 10 минут

🌵 **Прогнозируйте тренд акций с помощью Auto-ARIMA.**

**1. Выполните MSTL-декомпозицию ряда и извлеките тренд — дальше нужен только он.**

**2. Постройте прогноз вне выборки с Auto-ARIMA (не забудьте разделить ряд).**

In [ ]:
trend = res.trend
trend.plot()

# Регрессии

## Простая и множественная линейные регрессии

В простейшем случае **модель простой регрессии** описывает линейную связь между прогнозируемой переменной y и одним предиктором x:

$y_t = \beta_0 + \beta_1*x_t + \epsilon_t$

Здесь $y$ — зависимая переменная, а $x$ — независимая.

**Множественная линейная регрессия** похожа на простую, но использует несколько предикторов, среди которых могут быть лаги ряда или внешние признаки:

$y_t = \beta_0 + \beta_1*x_{1,t} + \beta_2*x_{2,t} + ... + \beta_k*x_{k,t} + \epsilon_t$

Коэффициенты $\beta_1, ..., \beta_k$ показывают вклад каждого предиктора с учётом остальных. То есть они отражают **предельные эффекты**: изменение зависимой переменной при изменении одного предиктора при прочих равных.

### Предпосылки

Используя линейную регрессию, мы неявно предполагаем следующее:

1. Модель разумно описывает реальность, то есть связь между целевой переменной и предикторами удовлетворяет линейному уравнению.

2. Ошибки $\epsilon_t$ должны:

* иметь среднее, равное нулю, иначе прогноз будет смещён;
* не обладать автокорреляцией, иначе в данных остаётся неиспользованная информация;
* быть независимыми от предикторов, иначе это влияние нужно включить в систематическую часть модели.

3. Каждый предиктор $x_t$ рассматривается как нерендомная величина. В контролируемом эксперименте мы задавали бы x и наблюдали y; в наблюдательных данных (бизнес, экономика) это невозможно, поэтому принимаем такое допущение.

4. Остатки модели должны быть стационарными, чтобы избежать фиктивных регрессий. Без стационарности модель может показывать ложные связи.


## Предикторы

### Dummy регрессоры

**Dummy регрессоры** помогают моделировать влияние особых событий на временной ряд: периодов, праздников, спецсобытий, изменений политики. Такие регрессоры ставят 1 в период события и 0 в остальных наблюдениях.

Главное преимущество dummy регрессоров — их легко продолжить в будущее (например, для Нового года можно заранее пометить все будущие даты в прогнозном горизонте).

**Торговая война США — Китай (2018–2020)**

* Старт: 6 июля 2018 года, когда США ввели тарифы на китайские товары объёмом 34 млрд долларов.

* Завершение: 15 января 2020 года — подписание первой фазы торговой сделки.

**Пандемия COVID-19 (2020 — настоящее время)**
* Старт: январь 2020 года, когда ВОЗ объявила чрезвычайную ситуацию.
* Последствия: несмотря на начало в 2020-м, экономический эффект ощущается и после 2020 года; многие страны испытывают его вплоть до 2024-го.

In [ ]:
date_range = pd.date_range(start=stocks.index.min(), end=stocks.index.max())

# Create the DataFrame
regressors = pd.DataFrame(date_range, columns=["ds"])

regressors

In [ ]:
# US-China trade war period
trade_war_start = "2018-07-06"
trade_war_end = "2020-01-15"

# Mark the trade war dates with 1 and other dates with 0
regressors['trade_war'] = ((regressors['ds'] >= trade_war_start) \
                            & (regressors['ds'] <= trade_war_end)).astype(int)

regressors = regressors[~regressors['ds'].dt.weekday.isin([5, 6])] # remove weekends
regressors.set_index('ds', inplace=True)

regressors = regressors.join(stocks).dropna()['trade_war'] # there're several nulls in stocks
regressors = pd.DataFrame(regressors)

regressors

Задание на 5 минут

🌵 **Напишите код для COVID-19-регрессора**

### Непрерывные регрессоры

Снова загрузим датасет по акциям Microsoft.

In [ ]:
stocks = pd.read_csv('microsoft.csv', parse_dates=['Date'])

In [ ]:
stocks.head()

Let's consider closing price as a continous numeric predictor.

In [ ]:
stocks.rename(columns={'Date':'ds', 'Close/Last':'y', 'Open':'x'}, inplace=True)
stocks['y'] = stocks['y'].apply(lambda x: x[1:]).astype(float)
stocks['x'] = stocks['x'].apply(lambda x: x[1:]).astype(float)

stocks = stocks[['ds','y', 'x']].sort_values(by='ds').set_index('ds')

In [ ]:
fig = px.line(title="Microsoft closing and open prices (daily)")
fig.add_scatter(x=stocks.index, y=stocks['y'], mode='lines', name='closing price', line=dict(color='blue'))
fig.add_scatter(x=stocks.index, y=stocks['x'], mode='lines', name='open price', line=dict(color='red'))

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()


# SUM UP

* Discussed different algorithms of reduction to stationarity: differencing, percent change, decompositions (classical, STL, MSTL) and importance of not autocorrelated residuals
* Discussed MLE and origin of AIC, BIC
* Discussed out-of-sample forecasting with ARMA, ARIMA and auto-ARIMA models
* Discussed regression models for time series
* Tried dummy regressors (or predictors) and continous regressors (or predictors)

Please, leave your feedback [here](https://forms.gle/5bhdQijqfojV8UF47), let's make our lessons more effective :)